条件分支示例


In [1]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

class WeatherState(TypedDict):
    temperature: int
    recommendation: str

def check_temperature(state: WeatherState) -> dict:
    # 这里可以调用真实的天气 API
    return {"temperature": 25}

def route_by_temperature(state: WeatherState) -> Literal["cold", "warm", "hot"]:
    """根据温度路由"""
    temp = state["temperature"]
    if temp < 15:
        return "cold"
    elif temp < 25:
        return "warm"
    else:
        return "hot"

def recommend_cold(state: WeatherState) -> dict:
    return {"recommendation": "穿厚外套！"}

def recommend_warm(state: WeatherState) -> dict:
    return {"recommendation": "穿长袖就好。"}

def recommend_hot(state: WeatherState) -> dict:
    return {"recommendation": "穿短袖，记得防晒！"}

# 构建图
graph = StateGraph(WeatherState)

graph.add_node("check", check_temperature)
graph.add_node("cold", recommend_cold)
graph.add_node("warm", recommend_warm)
graph.add_node("hot", recommend_hot)

graph.add_edge(START, "check")
graph.add_conditional_edges(
    "check",
    route_by_temperature,
    {
        "cold": "cold",
        "warm": "warm",
        "hot": "hot",
    }
)

graph.add_edge("cold", END)
graph.add_edge("warm", END)
graph.add_edge("hot", END)

app = graph.compile()

# 测试
result = app.invoke({})
print(f"温度：{result['temperature']}° C")
print(f"建议：{result['recommendation']}")
from IPython.display import Image, display

# 使用 Graphviz 渲染 (Colab 最稳定的方案)
try:
    display(Image(app.get_graph(xray=True).draw_png()))
except Exception as e:
    print(f"Graphviz 渲染失败: {e}")
    print("\n使用 Mermaid 文本方式显示:")
    print(app.get_graph(xray=True).draw_mermaid())

温度：25° C
建议：穿短袖，记得防晒！
Graphviz 渲染失败: Install pygraphviz to draw graphs: `pip install pygraphviz`.

使用 Mermaid 文本方式显示:
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	check(check)
	cold(cold)
	warm(warm)
	hot(hot)
	__end__([<p>__end__</p>]):::last
	__start__ --> check;
	check -.-> cold;
	check -.-> hot;
	check -.-> warm;
	cold --> __end__;
	hot --> __end__;
	warm --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

